## Measuring the Imbalance

In [98]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("cleaned_data.csv")

x = df.drop("default_payment_next_month", axis=1)
y = df["default_payment_next_month"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=42)

In [99]:
y_train.value_counts()

default_payment_next_month
0    18668
1     5304
Name: count, dtype: int64

In [100]:
y_train.value_counts()*100/(18668+5304)

default_payment_next_month
0    77.874187
1    22.125813
Name: count, dtype: float64

### What percentage of samples for Non-defaulters (class 0) and defaulters (class 1)
- Class 0: 77.87 %
- Class 1: 22.12 %

### What is the imbalance ratio?
- Imbalance Ratio ≈ 4 : 1

## Why Accuracy Fails Here

In [101]:
baseline_accuracy = (y_test == 0).mean()
baseline_accuracy

np.float64(0.7787418655097614)

### If the model predicts all non-default, what accuracy would it achieve?
- If model predicts all instances as non-defaulter accuracy would be 77.87%.

### Why is this misleading?
- Because although accuracy is high but we are misclassifying all the defaulters as non-defaulters which leads to nothing. we are creating this model to predict defaulter before they default so this accuracy as metric is very misleading.

## Choosing the Correct Evaluation Metrics

### Which metric is most important for credit default prediction?
- Recall is most important metric in credit default prediction

### Why is Recall of default class critical?
- Because recall tells us out of all the actual positive (defaults) how many did we predicted correctly and we want that all defaulters must be predicted correctly.

## Making Default XGBClassifier and Defining Baseline

In [102]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

numeric_feature = [
    "limit_bal", "age", "bill_amt1", "bill_amt2", "bill_amt3", "bill_amt4", "bill_amt5", "bill_amt6", "pay_amt1", "pay_amt2", "pay_amt3", "pay_amt4", "pay_amt5", "pay_amt6"
]

preprocessor = ColumnTransformer(transformers=[("num", StandardScaler(), numeric_feature)], remainder="passthrough")

In [103]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

xgb_default = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(
            n_estimators=400,
            max_depth=3,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42
        ))
    ]
)

xgb_default.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

In [104]:

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, recall_score, precision_score, f1_score

y_pred = xgb_default.predict(x_test)
y_prob = xgb_default.predict_proba(x_test)[ : ,1]

print("                    : Baseline : \n")
print(classification_report(y_test, y_pred))
print("Confusion Matrix : \n", confusion_matrix(y_test, y_pred))
print("ROC_AUC :", roc_auc_score(y_test, y_prob))
print("Recall :", recall_score(y_test, y_pred))
print("Precision Score :", precision_score(y_test, y_pred))
print("f1_score :", f1_score(y_test, y_pred))

                    : Baseline : 

              precision    recall  f1-score   support

           0       0.84      0.95      0.89      4667
           1       0.66      0.36      0.47      1326

    accuracy                           0.82      5993
   macro avg       0.75      0.66      0.68      5993
weighted avg       0.80      0.82      0.80      5993

Confusion Matrix : 
 [[4425  242]
 [ 846  480]]
ROC_AUC : 0.7724520969898402
Recall : 0.36199095022624433
Precision Score : 0.6648199445983379
f1_score : 0.46875


## Class Weighting Strategy

In [105]:
neg, pos = y_train.value_counts()
scale_pos_weight = neg/pos
scale_pos_weight

3.519607843137255

### Why is this ratio used?
- This ratio is used to handle imbalanced classification datasets by adjusting the balance between positive and negative class weights.

### What effect does increasing this weight have?
- It scales the gradient of the positive class (typically the minority class) during training, encouraging the model to pay more attention to correctly predicting the minority class.

In [106]:
from copy import deepcopy

xgb_weighted = deepcopy(xgb_default)

xgb_weighted.set_params(classifier__scale_pos_weight=scale_pos_weight)

xgb_weighted.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

In [107]:
wy_pred = xgb_weighted.predict(x_test)
wy_prob = xgb_weighted.predict_proba(x_test)[ : ,1]

print("                    : Weighted : \n")
print(classification_report(y_test, wy_pred))
print("Confusion Matrix : \n", confusion_matrix(y_test, wy_pred))
print("ROC_AUC :", roc_auc_score(y_test, wy_prob))
print("Recall :", recall_score(y_test, wy_pred))
print("Precision Score :", precision_score(y_test, wy_pred))
print("f1_score :", f1_score(y_test, wy_pred))

                    : Weighted : 

              precision    recall  f1-score   support

           0       0.88      0.79      0.83      4667
           1       0.46      0.63      0.53      1326

    accuracy                           0.76      5993
   macro avg       0.67      0.71      0.68      5993
weighted avg       0.79      0.76      0.77      5993

Confusion Matrix : 
 [[3696  971]
 [ 492  834]]
ROC_AUC : 0.7729232979803318
Recall : 0.6289592760180995
Precision Score : 0.46204986149584487
f1_score : 0.5327371446822101


In [108]:
import numpy as np
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_test, wy_prob)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

optimal_threshold  

np.float64(0.5067219734191895)

### Did recall increase?
- Yes, it went from 0.36 (Baseline) to 0.62 (Weighted).

### Did precision decrease?
- Yes, it went from 0.66 (Baseline) to 0.46 (Weighted).

## Oversampling the Minority Class (Using SMOTE)

In [109]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

x_resampled, y_resampled = smote.fit_resample(x_train, y_train)

### What is SMOTE?
- Synthetic Minority Over-sampling Technique (SMOTE) is a preprocessing method used to address class imbalance in dataset. it solves this by  It works by generating synthetic samples for the minority class instead of simply duplicating existing ones, which helps prevent overfitting. 

### What does SMOTE actually do?
- SMOTE generates new synthetic examples by interpolating between existing minority samples and their nearest neighbors in feature space. This increases the representation of the minority class while reducing the risk of overfitting that can occur with simple duplication.

- This synthetic samples are created along the line connecting similar minority instances, SMOTE helps the model learn a better decision boundary for the minority class.

### Why might it create unrealistic samples?
- Because it creates synthetic sample along the line connecting nearest sample in feature space without understanding real-world relation between those feature. 

In [110]:
xgb_smote = deepcopy(xgb_default)

xgb_smote.fit(x_resampled, y_resampled)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

In [111]:
sy_pred = xgb_smote.predict(x_test)
sy_prob = xgb_smote.predict_proba(x_test)[ : ,1]

print("                    : Over-sampled (SMOTE) : \n")
print(classification_report(y_test, sy_pred))
print("Confusion Matrix : \n", confusion_matrix(y_test, sy_pred))
print("ROC_AUC :", roc_auc_score(y_test, sy_prob))
print("Recall :", recall_score(y_test, sy_pred))
print("Precision Score :", precision_score(y_test, sy_pred))
print("f1_score :", f1_score(y_test, sy_pred))

                    : Over-sampled (SMOTE) : 

              precision    recall  f1-score   support

           0       0.85      0.85      0.85      4667
           1       0.47      0.48      0.47      1326

    accuracy                           0.76      5993
   macro avg       0.66      0.66      0.66      5993
weighted avg       0.77      0.76      0.76      5993

Confusion Matrix : 
 [[3949  718]
 [ 695  631]]
ROC_AUC : 0.7405091620798901
Recall : 0.475867269984917
Precision Score : 0.46775389177168275
f1_score : 0.47177570093457943


### Did recall increase?
- Yes, it went from 0.36 (Baseline) to 0.79 (over-sampled).

### Did precision collapse?
- Yes, it went from 0.66 (Baseline) to 0.31 (over-sampled).

## When Oversampling Hurts

### Why can SMOTE distort feature relationships?
- SMOTE generates synthetic samples by interpolating between a minority sample and one of its nearest neighbors in feature space. This process assumes that feature values can be meaningfully combined along a straight line. However, in many real-world datasets, features have constraints or nonlinear relationships. Interpolating between two valid samples may produce combinations of feature values that never occur naturally in the data, which can distort the underlying relationships between features and introduce unrealistic training points.

### Why might tree models already handle imbalance reasonably well?
- Tree-based models such as Random Forest and XGBoost can often handle class imbalance better than many other algorithms because they split the data into regions using decision rules. During training, trees can isolate minority class regions if informative features exist. Additionally, boosting algorithms can focus more on misclassified samples during later iterations, which naturally gives more attention to the minority class without requiring explicit oversampling.

## Undersampling Majority Class

In [112]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)

x_under, y_under = rus.fit_resample(x_train, y_train)

### What information might be lost here?
- As Undersampling removes instance of majority class till both class have equal number of instances. But in doing so model cannot see full diversity of patterns present thus losing subgroups, edge cases or rare pattern within the majority class. 

### When could this strategy still be useful?
- Undersampling can be useful when the dataset is very large and heavily imbalanced, because removing some majority samples will still leave enough data for the model to learn meaningful patterns. It is also helpful when the majority class contains many redundant or highly similar samples, since reducing them can simplify the dataset without losing much information.

In [113]:
xgb_under = deepcopy(xgb_default)

xgb_under.fit(x_under, y_under)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

In [114]:
uy_pred = xgb_under.predict(x_test)
uy_prob = xgb_under.predict_proba(x_test)[ : ,1]

print("                    : Under-sampled : \n")
print(classification_report(y_test, uy_pred))
print("Confusion Matrix : \n", confusion_matrix(y_test, uy_pred))
print("ROC_AUC :", roc_auc_score(y_test, uy_prob))
print("Recall :", recall_score(y_test, uy_pred))
print("Precision Score :", precision_score(y_test, uy_pred))
print("f1_score :", f1_score(y_test, uy_pred))

                    : Under-sampled : 

              precision    recall  f1-score   support

           0       0.88      0.78      0.83      4667
           1       0.45      0.64      0.53      1326

    accuracy                           0.75      5993
   macro avg       0.67      0.71      0.68      5993
weighted avg       0.79      0.75      0.76      5993

Confusion Matrix : 
 [[3626 1041]
 [ 480  846]]
ROC_AUC : 0.7666436075509797
Recall : 0.6380090497737556
Precision Score : 0.4483306836248013
f1_score : 0.5266106442577031


### Did recall increase?
- Yes, it went from 0.36 (Baseline) to 0.90 (under-sampled).

### Did precision collapse?
- Yes, it went from 0.66 (Baseline) to 0.27 (under-sampled).

## Comparing All Strategies

Model Variant | Precision | Recall |  F1  | ROC-AUC |
--------------|-----------|--------|------|---------|
Baseline      | 0.66      | 0.36   | 0.46 | 0.77    |
Class Weight  | 0.46      | 0.62   | 0.53 | 0.77    |
SMOTE         | 0.31      | 0.79   | 0.45 | 0.74    |
Undersampling | 0.27      | 0.90   | 0.41 | 0.76    |

### Which method improved minority recall the most?
- Undersampling improves minority recall the most.

### Which method produced the best overall balance?
- Class Weight provides best overall performance as it's f1 score is 0.53 which indicates that it provides best trade-off between recall and precision.

## Threshold Adjustment vs Resampling

In [115]:
thresholds = [0.3, 0.35, 0.5, 0.5067219734191895]

for threshold in thresholds:
    
    ty_prob = xgb_weighted.predict_proba(x_test)[:, 1]
    ty_pred = (ty_prob > threshold).astype(int)
    
    print(f"\n                    : Weighted (threshold = {threshold}) : \n")
    print("Confusion Matrix : \n", confusion_matrix(y_test, ty_pred))
    print("ROC_AUC :", roc_auc_score(y_test, ty_prob))
    print("Recall :", recall_score(y_test, ty_pred))
    print("Precision Score :", precision_score(y_test, ty_pred))
    print("f1_score :", f1_score(y_test, ty_pred))
        


                    : Weighted (threshold = 0.3) : 

Confusion Matrix : 
 [[2174 2493]
 [ 207 1119]]
ROC_AUC : 0.7729232979803318
Recall : 0.8438914027149321
Precision Score : 0.30980066445182725
f1_score : 0.4532199270959903

                    : Weighted (threshold = 0.35) : 

Confusion Matrix : 
 [[2662 2005]
 [ 282 1044]]
ROC_AUC : 0.7729232979803318
Recall : 0.7873303167420814
Precision Score : 0.34240734667103967
f1_score : 0.4772571428571429

                    : Weighted (threshold = 0.5) : 

Confusion Matrix : 
 [[3696  971]
 [ 492  834]]
ROC_AUC : 0.7729232979803318
Recall : 0.6289592760180995
Precision Score : 0.46204986149584487
f1_score : 0.5327371446822101

                    : Weighted (threshold = 0.5067219734191895) : 

Confusion Matrix : 
 [[3732  935]
 [ 501  825]]
ROC_AUC : 0.7729232979803318
Recall : 0.6221719457013575
Precision Score : 0.46875
f1_score : 0.5346727154893065


### How does lowering threshold affect recall?
- Lowering threshold increases recall.

### Does threshold tuning achieve similar results to SMOTE?
- Yes, by setting threshold to 0.35 weight balanced model achieves very similar result to SMOTE sample model. 

## Strategy Selection

### Which approach should be preferred for this dataset?
- By experimenting i do say weight balancing + threshold tuning is best option because along with recall being high and it being pretty balanced it's ROC-AUC score is high too which means that model will give reliable result.

- Based on the experiments, class weighting combined with threshold tuning appears to be the most effective strategy for this dataset. Class weighting improves the model’s ability to detect the minority class during training, while threshold tuning allows control over the precision–recall tradeoff at prediction time. Compared to SMOTE and undersampling, this approach achieves a more balanced performance with a higher F1 score while maintaining a strong ROC-AUC value. This suggests the model preserves good ranking ability while improving recall for the minority class without introducing excessive false positives.